# Optimizing `min_signal_frac`

The registration gate (`enough_signal` in `registration.py`) skips constrained PCC on a seam when **either** overlap strip has less than `min_signal_frac` of its pixels above the tissue threshold, falling back to the nominal overlap (`dy=dx=0`). The default cutoff (`0.05`) was picked by hand.

This notebook picks it from data instead. The idea (per the plan):

1. We **already** compute the per-strip signal fraction while building the edge-focus box for PCC (`enough_signal` returns `frac_a`, `frac_b`). So record those values for every seam.
2. On every seam, run PCC **unconditionally** (gate disabled) and score the result with an *independent* witness — raw MSE / MI, naive vs registered, inside the same tissue box the QC uses. This tells us whether PCC actually **helped** or **hurt** that seam.
3. Look at the distribution of signal fraction split by helped / hurt, then **sweep the cutoff** and see where it stops admitting the failures without throwing away the wins. Set the cutoff there.

The failure mode we are gating out: on a near-empty overlap, PCC latches onto a spurious peak, produces a real shift, and makes agreement *worse* than just butting the tiles at the nominal overlap. A good cutoff excludes those seams and keeps the ones where PCC improves agreement.

> Run under the project kernel (`/opt/miniconda3/bin/python`). All registration decisions are imported from `registration.py` / `registrationQC.py` — nothing is re-implemented — so what we characterize here is exactly what the pipeline does.

## Parameters

Point `INPUT_DIRS` at one or more tile folders (each a single acquisition named `<stem>[<r> x <c>]_C<ch>_z<z>.ome.tif`). Pooling several acquisitions gives a richer distribution. `MAX_SLICES_PER_DIR` evenly subsamples the z-stack so a full volume stays tractable; set to `None` to use every detected slice. Algorithm knobs mirror `registration.py`'s defaults so the box, high-pass and constrained search match the real run.

In [1]:
from types import SimpleNamespace
from pathlib import Path

# --- What to run on -------------------------------------------------
INPUT_DIRS = [
    '/Users/spaltahill/test_images',
    # add more acquisition folders here to pool their seams into one distribution
]
REFERENCE_CHANNEL = 0          # the channel registration.py registers on
MAX_SLICES_PER_DIR = None      # int -> evenly subsample the z-stack; None -> all slices
OUTPUT_DIR = 'qc_results/min_signal_optimization'

# --- Algorithm knobs (defaults match registration.py) --------------
p = SimpleNamespace(
    overlap_frac_h=0.20,
    overlap_frac_v=0.20,
    otsu_multiplier=0.5,
    edge_pad_px=80,
    highpass_sigma=12.0,
    max_cross_px=20,
    join_band_frac=1.0,
)

# --- Cutoff study knobs --------------------------------------------
CURRENT_DEFAULT = 0.05                 # the hand-picked cutoff we are testing
CANDIDATE_CUTOFFS = None               # None -> auto grid; or e.g. np.linspace(0, 0.3, 61)
# A seam counts as HELPED / HURT when registration moves raw MI by more than this many
# nats vs the naive placement; within +/- it is a NEUTRAL no-op. Kept small; the
# continuous scatter below makes the exact value non-load-bearing.
MI_TOL = 0.005

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print('output ->', Path(OUTPUT_DIR).resolve())

output -> /Users/spaltahill/microscopy-stitcher/qc_results/min_signal_optimization


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.filters import threshold_otsu

# Every registration decision is imported, never re-implemented.
from registration import (edge_focus_box, register_pcc_constrained, enough_signal,
                          load_grid, scan_input_folder, composite_line)
from registrationQC import mutual_information, _overlap_box_h, _overlap_box_v

plt.rcParams['figure.dpi'] = 110

## Per-seam analysis

`analyze_seam` runs the exact box + constrained-PCC the engine runs, but with **no gate**, and scores naive vs registered inside the same masked tissue box `registrationQC.score_seam` uses (raw MSE / MI, foreground pixels only). It returns one record per seam. `min_frac = min(frac_a, frac_b)` is the quantity the gate actually thresholds on.

In [3]:
def _score(a, b, axis, overlap, box_lo, box_hi, dy, dx, thresh):
    """Raw MSE / MI naive-vs-registered inside the tissue box, mirroring
    registrationQC.score_seam's metric math (no plotting). Returns
    (mse_n, mse_r, mi_n, mi_r, nfg_r)."""
    if axis == 'h':
        nA, nB, rA, rB = _overlap_box_h(a, b, overlap, box_lo, box_hi, dy, dx)
    else:
        nA, nB, rA, rB = _overlap_box_v(a, b, overlap, box_lo, box_hi, dy, dx)

    def crop(arr, box):
        r0, r1, c0, c1 = box
        return arr[r0:r1, c0:c1]

    out = {}
    for tag, boxA, boxB in [('n', nA, nB), ('r', rA, rB)]:
        ra, rb = crop(a, boxA), crop(b, boxB)
        hh = min(ra.shape[0], rb.shape[0])
        ww = min(ra.shape[1], rb.shape[1])
        if hh > 0 and ww > 0:
            mask = (ra[:hh, :ww] > thresh) | (rb[:hh, :ww] > thresh)
        else:
            mask = np.zeros((0, 0), dtype=bool)
        if mask.sum() >= 2:
            pa = ra[:hh, :ww][mask].astype('float32')
            pb = rb[:hh, :ww][mask].astype('float32')
            out[f'mse_{tag}'] = float(np.mean((pa - pb) ** 2, dtype='float64'))
            out[f'mi_{tag}'] = mutual_information(pa, pb)
        else:
            out[f'mse_{tag}'] = out[f'mi_{tag}'] = float('nan')
        out[f'nfg_{tag}'] = int(mask.sum())
    return out['mse_n'], out['mse_r'], out['mi_n'], out['mi_r'], out['nfg_r']


def analyze_seam(a, b, axis, overlap, thresh):
    """One seam, gate DISABLED (PCC always runs). Returns the record dict and the
    (dy, dx, overlap) shift so callers can build the registered composite."""
    if axis == 'h':
        sa, sb = a[:, -overlap:], b[:, :overlap]
        box_axis = 0
    else:
        sa, sb = a[-overlap:, :], b[:overlap, :]
        box_axis = 1
    _, fa, fb = enough_signal(sa, sb, thresh, 0.0)          # same frac the gate reads
    lo, hi = edge_focus_box(sa, sb, thresh, p.edge_pad_px, axis=box_axis)
    join_half = int(round(p.join_band_frac * overlap))
    box = (sa[lo:hi], sb[lo:hi]) if axis == 'h' else (sa[:, lo:hi], sb[:, lo:hi])
    dy, dx = register_pcc_constrained(box[0], box[1], p.highpass_sigma, True,
                                      p.max_cross_px, 0, join_half, axis)
    mse_n, mse_r, mi_n, mi_r, nfg = _score(a, b, axis, overlap, lo, hi, dy, dx, thresh)
    rec = dict(axis=axis, overlap=overlap, box_span=hi - lo,
               frac_a=fa, frac_b=fb, min_frac=min(fa, fb), max_frac=max(fa, fb),
               dy=dy, dx=dx, shift_mag=float(np.hypot(dy, dx)),
               mse_n=mse_n, mse_r=mse_r, mi_n=mi_n, mi_r=mi_r, nfg=nfg)
    return rec, (dy, dx, overlap)

## Sweep the data

For each acquisition and each selected z-slice: load the reference-channel grid, set the Otsu tissue threshold the pipeline uses, then record every horizontal seam (per grid row) and every vertical seam (between the horizontally-stitched rows). Rows are composited with the unconstrained (always-PCC) shifts so the vertical seams see the best-aligned rows.

In [4]:
def select_slices(z_slices, k):
    """k z-slices evenly spaced by position in the sorted stack (None -> all)."""
    if k is None or k >= len(z_slices):
        return list(z_slices)
    idx = np.round(np.linspace(0, len(z_slices) - 1, k)).astype(int)
    return [z_slices[i] for i in sorted(set(idx))]


records = []
for input_dir in INPUT_DIRS:
    stem, n_rows, n_cols, channels, z_slices, index = scan_input_folder(input_dir)
    if REFERENCE_CHANNEL not in channels:
        raise SystemExit(f'{input_dir}: reference channel {REFERENCE_CHANNEL} not in {channels}')
    picks = select_slices(z_slices, MAX_SLICES_PER_DIR)
    acq = Path(input_dir).name
    print(f'{acq}: {n_rows}x{n_cols} grid, {len(z_slices)} z-slices, using {len(picks)}')

    for z in picks:
        grid = load_grid(index, REFERENCE_CHANNEL, z, n_rows, n_cols)
        sample = np.concatenate([t[::8, ::8].ravel() for row in grid for t in row])
        thresh = p.otsu_multiplier * threshold_otsu(sample)

        # Stage 1: horizontal seams, and build each row's registered composite.
        rows_reg = []
        for r, tiles in enumerate(grid):
            h_shifts = []
            for i in range(1, len(tiles)):
                a, b = tiles[i - 1], tiles[i]
                overlap = max(1, min(int(round(p.overlap_frac_h * a.shape[1])),
                                     a.shape[1], b.shape[1]))
                rec, shift = analyze_seam(a, b, 'h', overlap, thresh)
                rec.update(acq=acq, z=z, seam=f'R{r} H{i-1}-{i}')
                records.append(rec)
                h_shifts.append(shift)
            rows_reg.append(composite_line(tiles, h_shifts, 'h', p.overlap_frac_h,
                                           register=True))

        # Stage 2: vertical seams between the stitched rows.
        for i in range(1, len(rows_reg)):
            a, b = rows_reg[i - 1], rows_reg[i]
            overlap = max(1, min(int(round(p.overlap_frac_v * a.shape[0])),
                                 a.shape[0], b.shape[0]))
            rec, _ = analyze_seam(a, b, 'v', overlap, thresh)
            rec.update(acq=acq, z=z, seam=f'V {i-1}-{i}')
            records.append(rec)
        print(f'  z{z:04d}: thresh={thresh:.1f}  seams so far={len(records)}')

df = pd.DataFrame(records)
print(f'\nTotal seams collected: {len(df)}')
df.head()

test_images: 4x3 grid, 6 z-slices, using 6


  z0100: thresh=1281.3  seams so far=11


  z0101: thresh=1273.3  seams so far=22


  z0102: thresh=1273.7  seams so far=33


  z0190: thresh=1377.1  seams so far=44


  z0191: thresh=1389.0  seams so far=55


  z0192: thresh=1393.7  seams so far=66

Total seams collected: 66


,axis,overlap,box_span,frac_a,frac_b,min_frac,max_frac,dy,dx,shift_mag,mse_n,mse_r,mi_n,mi_r,nfg,acq,z,seam
0,h,432,608,0.004367,0.033679,0.004367,0.033679,-5.990101,54.971444,55.296844,122710.010465,72824.063943,0.131734,0.333820,16546,test_images,100,R0 H0-1
1,h,432,807,0.112929,0.073293,0.073293,0.112929,16.020310,21.997271,27.212686,125155.424595,144102.263904,0.676026,0.648479,127088,test_images,100,R0 H1-2
2,h,432,2491,0.772945,0.873538,0.772945,0.873538,2.115106,-63.459543,63.494781,588074.616037,829050.872594,1.379418,1.441373,1116803,test_images,100,R1 H0-1
3,h,432,2459,0.871696,0.851958,0.851958,0.871696,-5.931707,-66.184408,66.449688,599385.897718,676677.343472,0.917717,1.020908,1119752,test_images,100,R1 H1-2
4,h,432,2560,0.938638,0.995222,0.938638,0.995222,-3.883402,-67.904868,68.015821,456160.007514,346209.844061,1.093689,1.231016,1272950,test_images,100,R2 H0-1


## Label each seam: did PCC help or hurt?

Raw MI is the trustworthy witness (raw MSE also carries the per-tile exposure gap). `mi_gain = mi_r - mi_n`: positive means the measured shift agrees better than the nominal overlap. A seam is **HELPED** when `mi_gain > +MI_TOL`, **HURT** when `mi_gain < -MI_TOL` (the failures we want to gate out), and **NEUTRAL** in between (PCC found essentially the nominal placement — harmless).

In [5]:
df['mi_gain'] = df['mi_r'] - df['mi_n']
df['mse_pct'] = 100.0 * (df['mse_r'] - df['mse_n']) / df['mse_n']   # negative = better

def outcome(g):
    if g > MI_TOL:
        return 'helped'
    if g < -MI_TOL:
        return 'hurt'
    return 'neutral'

df['outcome'] = df['mi_gain'].apply(outcome)
OC_COLOR = {'helped': '#4c72b0', 'neutral': '#999999', 'hurt': '#c44e52'}

print(df['outcome'].value_counts())
print('\nmin_frac by outcome:')
print(df.groupby('outcome')['min_frac'].describe()[['count', 'min', '25%', '50%', '75%', 'max']])
df.to_csv(Path(OUTPUT_DIR) / 'seam_records.csv', index=False)
print('\nsaved', Path(OUTPUT_DIR) / 'seam_records.csv')

outcome
helped     51
neutral     9
hurt        6
Name: count, dtype: int64

min_frac by outcome:
         count       min       25%       50%       75%       max
outcome                                                         
helped    51.0  0.004367  0.330984  0.457558  0.777103  0.999987
hurt       6.0  0.073293  0.173771  0.502887  0.559781  0.560584
neutral    9.0  0.000000  0.000000  0.429746  0.998686  0.998752

saved qc_results/min_signal_optimization/seam_records.csv


## Where do failures live in signal space?

If the hypothesis holds, HURT seams cluster at low `min_frac` and HELPED seams sit higher. The vertical lines mark the current default (`0.05`) and, once computed below, the recommended cutoff.

In [6]:
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(15, 5))

# (a) stacked histogram of min_frac by outcome
bins = np.linspace(0, min(0.5, df['min_frac'].max() * 1.05 + 1e-3), 40)
ax0.hist([df.loc[df.outcome == o, 'min_frac'] for o in ['helped', 'neutral', 'hurt']],
         bins=bins, stacked=True, label=['helped', 'neutral', 'hurt'],
         color=[OC_COLOR[o] for o in ['helped', 'neutral', 'hurt']])
ax0.axvline(CURRENT_DEFAULT, color='k', ls='--', lw=1.5, label=f'default {CURRENT_DEFAULT}')
ax0.set_xlabel('min_frac  (min of the two strip signal fractions)')
ax0.set_ylabel('# seams')
ax0.set_title('Signal fraction distribution, split by PCC outcome')
ax0.legend()

# (b) the money plot: min_frac vs MI gain, colored by outcome, sized by
# foreground pixels (big dots = trustworthy MI; tiny dots = few px, noisy).
sz = 12 + 90 * np.sqrt(df['nfg'] / df['nfg'].max())
for o in ['helped', 'neutral', 'hurt']:
    m = df.outcome == o
    ax1.scatter(df.loc[m, 'min_frac'], df.loc[m, 'mi_gain'], s=sz[m], alpha=0.7,
                color=OC_COLOR[o], label=o, edgecolor='none')
ax1.axhline(0, color='k', lw=0.8)
ax1.axhline(MI_TOL, color='gray', ls=':', lw=0.8)
ax1.axhline(-MI_TOL, color='gray', ls=':', lw=0.8)
ax1.axvline(CURRENT_DEFAULT, color='k', ls='--', lw=1.5, label=f'default {CURRENT_DEFAULT}')
ax1.set_xlabel('min_frac')
ax1.set_ylabel('MI gain  (registered − naive, nats)')
ax1.set_title('PCC benefit vs available signal  (dot size ~ foreground px)')
ax1.legend()
fig.tight_layout()
fig.savefig(Path(OUTPUT_DIR) / 'signal_vs_outcome.png', bbox_inches='tight')
plt.show()

/var/folders/8k/w9mhn1zn6_7_2ls7jxy732kr0000gn/T/ipykernel_71229/3050648802.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Sweep the cutoff

Simulate the gate at every candidate cutoff `c`: a seam with `min_frac >= c` keeps its **registered** result; below `c` it falls back to **naive**. Two readouts:

- **Net metric** — foreground-pixel-weighted mean MI (and MSE) over all seams under that gate. Higher MI / lower MSE is better. The best cutoff is the one that keeps the wins and drops the failures.
- **Admission counts** — how many HELPED vs HURT seams the gate still lets through PCC. We want to admit HELPED and reject HURT.

Recommended cutoff = the `c` that maximizes net MI (smallest such `c` on ties).

In [7]:
if CANDIDATE_CUTOFFS is None:
    hi = float(np.nanpercentile(df['min_frac'], 99)) if len(df) else 0.3
    cutoffs = np.linspace(0.0, max(0.3, hi), 61)
else:
    cutoffs = np.asarray(CANDIDATE_CUTOFFS, dtype=float)

w = df['nfg'].to_numpy(dtype=float)                       # weight seams by foreground pixels
mi_r, mi_n = df['mi_r'].to_numpy(), df['mi_n'].to_numpy()
mse_r, mse_n = df['mse_r'].to_numpy(), df['mse_n'].to_numpy()
mf = df['min_frac'].to_numpy()
oc = df['outcome'].to_numpy()

def weighted_mean(vals, weights):
    m = np.isfinite(vals) & np.isfinite(weights)
    return float(np.sum(vals[m] * weights[m]) / np.sum(weights[m])) if weights[m].sum() else np.nan

rows = []
for c in cutoffs:
    run = mf >= c                                         # PCC runs; else fall back to naive
    net_mi = weighted_mean(np.where(run, mi_r, mi_n), w)
    net_mse = weighted_mean(np.where(run, mse_r, mse_n), w)
    rows.append(dict(cutoff=c, net_mi=net_mi, net_mse=net_mse,
                     helped_admitted=int(np.sum(run & (oc == 'helped'))),
                     hurt_admitted=int(np.sum(run & (oc == 'hurt'))),
                     helped_total=int(np.sum(oc == 'helped')),
                     hurt_total=int(np.sum(oc == 'hurt'))))
sweep = pd.DataFrame(rows)

best_i = int(np.nanargmax(sweep['net_mi'].to_numpy()))
best_cut = float(sweep['cutoff'].iloc[best_i])
sweep.to_csv(Path(OUTPUT_DIR) / 'cutoff_sweep.csv', index=False)
print(f'Recommended cutoff (max net MI): {best_cut:.3f}')
print(f'  net MI  @ recommended {sweep.net_mi.iloc[best_i]:.4f}  vs @ default '
      f'{np.interp(CURRENT_DEFAULT, sweep.cutoff, sweep.net_mi):.4f}')

Recommended cutoff (max net MI): 0.083
  net MI  @ recommended 0.8976  vs @ default 0.8974


In [8]:
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(15, 5))

# net metric vs cutoff (MI left axis, MSE right axis)
ax0.plot(sweep.cutoff, sweep.net_mi, color='#4c72b0', label='net MI (higher better)')
ax0.set_xlabel('cutoff  (min_signal_frac)')
ax0.set_ylabel('net MI  [nats]', color='#4c72b0')
ax0.tick_params(axis='y', labelcolor='#4c72b0')
axb = ax0.twinx()
axb.plot(sweep.cutoff, sweep.net_mse, color='#c44e52', label='net MSE (lower better)')
axb.set_ylabel('net MSE  [raw]', color='#c44e52')
axb.tick_params(axis='y', labelcolor='#c44e52')
ax0.axvline(best_cut, color='green', lw=2, label=f'recommended {best_cut:.3f}')
ax0.axvline(CURRENT_DEFAULT, color='k', ls='--', lw=1.5, label=f'default {CURRENT_DEFAULT}')
ax0.set_title('Net agreement under the gate vs cutoff')
ax0.legend(loc='lower left', fontsize=8)

# admission counts vs cutoff
ax1.plot(sweep.cutoff, sweep.helped_admitted, color='#4c72b0', label='HELPED admitted (keep high)')
ax1.plot(sweep.cutoff, sweep.hurt_admitted, color='#c44e52', label='HURT admitted (drive to 0)')
ax1.axvline(best_cut, color='green', lw=2, label=f'recommended {best_cut:.3f}')
ax1.axvline(CURRENT_DEFAULT, color='k', ls='--', lw=1.5, label=f'default {CURRENT_DEFAULT}')
ax1.set_xlabel('cutoff  (min_signal_frac)')
ax1.set_ylabel('# seams admitted to PCC')
ax1.set_title('What the gate lets through vs cutoff')
ax1.legend(fontsize=8)
fig.tight_layout()
fig.savefig(Path(OUTPUT_DIR) / 'cutoff_sweep.png', bbox_inches='tight')
plt.show()

/var/folders/8k/w9mhn1zn6_7_2ls7jxy732kr0000gn/T/ipykernel_71229/2098333400.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Recommendation

In [9]:
hurt = df[df.outcome == 'hurt']
helped = df[df.outcome == 'helped']
# A robust cutoff also sits just above where failures live: the max min_frac among HURT
# seams (so all observed failures are excluded), unless that would drop most wins.
hurt_ceiling = float(hurt['min_frac'].max()) if len(hurt) else float('nan')
helped_kept_at_best = int(np.sum(helped['min_frac'].to_numpy() >= best_cut))

print('=' * 68)
print(f'  Seams analyzed        : {len(df)}  ({len(INPUT_DIRS)} acquisition(s))')
print(f'  helped / neutral / hurt: {(df.outcome=="helped").sum()} / '
      f'{(df.outcome=="neutral").sum()} / {(df.outcome=="hurt").sum()}')
print('  ' + '-' * 64)
print(f'  Current default cutoff : {CURRENT_DEFAULT}')
print(f'  Recommended (max netMI): {best_cut:.3f}')
if np.isfinite(hurt_ceiling):
    print(f'  Highest min_frac among HURT seams: {hurt_ceiling:.3f}  '
          f'(a cutoff above this excludes every observed failure)')
print(f'  HELPED seams still admitted at recommended cutoff: '
      f'{helped_kept_at_best}/{len(helped)}')
print(f'  HURT seams admitted at recommended cutoff        : '
      f'{int(np.sum(hurt["min_frac"].to_numpy() >= best_cut))}/{len(hurt)}')
print('=' * 68)
print(f'\n  Run the pipeline with:  --min-signal-frac {best_cut:.3f}')
print('  (Re-run pooling more acquisitions/slices before committing to a value;')
print('   a handful of local slices may not exercise the low-signal tail.)')

  Seams analyzed        : 66  (1 acquisition(s))
  helped / neutral / hurt: 51 / 9 / 6
  ----------------------------------------------------------------
  Current default cutoff : 0.05
  Recommended (max netMI): 0.083
  Highest min_frac among HURT seams: 0.561  (a cutoff above this excludes every observed failure)
  HELPED seams still admitted at recommended cutoff: 48/51
  HURT seams admitted at recommended cutoff        : 4/6

  Run the pipeline with:  --min-signal-frac 0.083
  (Re-run pooling more acquisitions/slices before committing to a value;
   a handful of local slices may not exercise the low-signal tail.)
